# 03 — M0_hard: Hard Subset Training

Trains M0_hard on the KITTI Hard difficulty tier only, then evaluates on all three tiers.

**Hard tier criteria (ALL must be satisfied — Correction 5):**
- `height_px >= 25` (not just occlusion_lvl=2)
- `occlusion_lvl <= 2` (levels 0, 1, and 2 inclusive)
- `truncation <= 0.50`

**Expected result:** M0_hard trained on hard-tier instances shows a performance gap vs M0 trained on all data — because the Hard tier contains genuinely harder detection problems.

**Prerequisite:** notebook `01_baseline` must have completed.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
FIGURES_DIR = ROOT / 'results' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = 'local'   # change to 'kaggle' for full training
SMOKE_TEST = (RUN_MODE == 'local')

cfg = load_config(ROOT / 'configs' / 'm0_baseline.yaml', overrides={
    'run_mode': RUN_MODE,
    'model_id': 'M0_hard',
    'epochs': 3 if SMOKE_TEST else 100,
    'data_limit': 100 if SMOKE_TEST else None,
})
cfg.ensure_dirs()
set_all_seeds(cfg.seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Model: {cfg.model_id}  |  Mode: {cfg.mode}  |  Device: {device}')

In [ ]:
# ── Filter training set to KITTI Hard tier ────────────────────────────────────
# Hard tier: height_px >= 25 AND occlusion_lvl <= 2 AND truncation <= 0.50
# This is NOT just occlusion_lvl == 2 (Correction 5).

full_train_ds = KITTIDataset(
    cfg.kitti_root, split='train', imgsz=cfg.imgsz,
    data_limit=cfg.data_limit
)

def image_qualifies_for_hard_tier(sample):
    """
    Returns True if the image contains at least one annotation that meets
    the KITTI Hard tier criteria:
      height_px >= 25, occlusion_lvl <= 2, truncation <= 0.50
    """
    if len(sample['height_px']) == 0:
        return False
    height_ok = sample['height_px'] >= 25.0
    occ_ok    = sample['occlusion_lvl'] <= 2
    trunc_ok  = sample['truncation'] <= 0.50
    return bool((height_ok & occ_ok & trunc_ok).any())

hard_indices = [
    i for i in range(len(full_train_ds))
    if image_qualifies_for_hard_tier(full_train_ds[i])
]

hard_train_ds = Subset(full_train_ds, hard_indices)
print(f'Full train set:      {len(full_train_ds)} images')
print(f'Hard tier subset:    {len(hard_train_ds)} images ({100*len(hard_train_ds)/len(full_train_ds):.1f}%)')

In [ ]:
# ── Train M0_hard ──────────────────────────────────────────────────────────────
from ultralytics import YOLO

val_ds = KITTIDataset(cfg.kitti_root, split='val', imgsz=cfg.imgsz)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch, shuffle=False,
    collate_fn=collate_fn, num_workers=0
)

model = YOLO('yolov8s.pt')
logger = ExperimentLogger(
    run_id=f'M0_hard_seed{cfg.seed}',
    log_dir=cfg.log_dir,
    config={'model_id': 'M0_hard', 'hard_subset_size': len(hard_train_ds)},
)

with logger:
    # Train on hard-tier subset only
    # Note: Ultralytics uses its own data pipeline; we pass a filtered dataset YAML
    # or use the custom KITTIDataset loader via a custom trainer callback.
    # For the hard subset, we use the standard training path with a hard_tier flag.
    model.train(
        data=str(ROOT / 'data' / 'kitti_hard_ultralytics.yaml'),  # hard-tier filtered dataset config
        epochs=cfg.epochs,
        imgsz=cfg.imgsz,
        batch=cfg.batch,
        optimizer=cfg.optimizer,
        lr0=cfg.lr0,
        lrf=cfg.lrf,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
        warmup_epochs=cfg.warmup_epochs,
        amp=cfg.amp,
        seed=cfg.seed,
        project=str(cfg.checkpoint_dir),
        name='M0_hard',
        exist_ok=True,
        device=device,
    )

    best_ckpt = cfg.checkpoint_dir / 'M0_hard' / 'weights' / 'best.pt'
    best_model = YOLO(str(best_ckpt))
    best_model.model.eval()

    predictions, annotations = [], []
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image'].to(device)
            results = best_model(imgs, verbose=False)
            for i, r in enumerate(results):
                boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                predictions.append({'image_id': batch['image_id'][i], 'boxes': boxes, 'scores': scores})
                annotations.append(sample_to_annotation(batch, i))

    m0h_easy = compute_kitti_ap(predictions, annotations, 'easy')
    m0h_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
    m0h_hard = compute_kitti_ap(predictions, annotations, 'hard')
    logger.log_metrics({'AP_easy': m0h_easy, 'AP_mod': m0h_mod, 'AP_hard': m0h_hard})

print(f'M0_hard results:')
print(f'  AP_easy:  {m0h_easy:.2f}')
print(f'  AP_mod:   {m0h_mod:.2f}')
print(f'  AP_hard:  {m0h_hard:.2f}')

In [ ]:
# ── Load M0 results for comparison ───────────────────────────────────────────
results_csv = TABLES_DIR / 'val_results_all_models.csv'
if results_csv.exists():
    df_all = pd.read_csv(results_csv)
    m0_row = df_all[df_all['model_id'] == 'M0']
    if len(m0_row) > 0:
        m0_easy = float(m0_row['AP_easy'].values[0])
        m0_mod  = float(m0_row['AP_mod'].values[0])
        m0_hard = float(m0_row['AP_hard'].values[0])
    else:
        print('[WARN] M0 results not found. Run notebook 01 first.')
        m0_easy = m0_mod = m0_hard = None
else:
    print('[WARN] val_results_all_models.csv not found. Run notebook 01 first.')
    m0_easy = m0_mod = m0_hard = None

In [ ]:
# ── Plot M0 vs M0_hard ────────────────────────────────────────────────────────
tiers = ['Easy', 'Moderate', 'Hard']

fig, ax = plt.subplots(figsize=(7, 4))

if m0_hard is not None:
    ax.plot(tiers, [m0_easy, m0_mod, m0_hard],
            marker='o', linewidth=2, color='#2196F3', label='M0 (all data)')

ax.plot(tiers, [m0h_easy, m0h_mod, m0h_hard],
        marker='s', linewidth=2, linestyle='--', color='#F44336', label='M0_hard (hard-tier data only)')

ax.set_xlabel('Difficulty Tier')
ax.set_ylabel('AP (%)')
ax.set_title('M0 vs M0_hard: AP by Difficulty Tier')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 100)

# Annotate AP_hard values
if m0_hard is not None:
    ax.annotate(f'{m0_hard:.1f}', (tiers[2], m0_hard), textcoords='offset points',
                xytext=(8, 4), fontsize=9, color='#2196F3')
ax.annotate(f'{m0h_hard:.1f}', (tiers[2], m0h_hard), textcoords='offset points',
            xytext=(8, 4), fontsize=9, color='#F44336')

plt.tight_layout()
out = FIGURES_DIR / 'hard_subset_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.savefig(str(out).replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Append M0_hard to results table ──────────────────────────────────────────
row = {
    'model_id': 'M0_hard',
    'description': 'M0 trained on Hard tier subset only (h>=25, occ<=2, trunc<=0.50)',
    'AP_easy': m0h_easy,
    'AP_mod': m0h_mod,
    'AP_hard': m0h_hard,
    'aug_p': 0.0,
    'use_fem': False,
    'use_popam': False,
    'fusion_type': 'none',
    'use_vis_head': False,
    'seed': cfg.seed,
    'epochs': cfg.epochs,
}

if results_csv.exists():
    df_all = pd.read_csv(results_csv)
    df_all = df_all[df_all['model_id'] != 'M0_hard']
    df_all = pd.concat([df_all, pd.DataFrame([row])], ignore_index=True)
else:
    df_all = pd.DataFrame([row])

df_all.to_csv(results_csv, index=False)
print(f'Saved to: {results_csv}')
df_all